# 01 — The gen2 algebra line, end to end

The first notebook of the gen2 set, aligned with the rewritten reference
line (`hllset-next-v2`):

1. **The hinge** — `BitAddress` from `hllset-contracts`.
2. **The LUT lattice** — `LutNode`/`LutIndex`/`K_i` from `hllset-lut`.
3. **Ingest** — complete, single-touch, in-module (`hllset-morphisms`).
4. **Materialize** — LUT-first, TF only for ambiguity.
5. **The Noether context** — `Context`/`evolve`/tropical `FollowMatrix`.
6. **The five-level rank algebra** — TF stored, rank derived.
7. **The finest symmetry** — `HLLSet(LUT) = HLLSet(MerkleTree)`.


In [2]:
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-contracts" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-core" }
:dep hllset-lut = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-lut" }
:dep hllset-morphisms = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-morphisms" }
:dep hllset-context = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-context" }
:dep hllset-ranks = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-ranks" }


In [3]:
use hllset_contracts::{token_in_bytes, token_in_bytes_le, BitAddress, BITS_PER_REG, M};
use hllset_context::{evolve, invariants_hold, Context, FollowMatrix};
use hllset_core::HLLSet;
use hllset_lut::{AtomTree, LutIndex, LutNode};
use hllset_morphisms::{materialize, Ingest, SEEDS};
use hllset_ranks::{bit_rank, compound_rank, hllset_rank, register_rank, token_rank, Aggregator};
println!("gen2 algebra crates loaded: contracts, core, lut, morphisms, context, ranks");


gen2 algebra crates loaded: contracts, core, lut, morphisms, context, ranks


---
## 1. The hinge: BitAddress

The one object both morphisms pass through. Flat `u32` storage,
`reg()`/`tz()` accessors, bitmap order, and the soldered collision pin.


In [4]:
let addr = BitAddress::from_reg_tz(759, 0);
println!("759,0 -> bit {} (reg {}, tz {})", addr.bit(), addr.reg(), addr.tz());
println!("geometry: P={} M={} bits/reg={}", 10, M, BITS_PER_REG);
let a = BitAddress::of_token(&token_in_bytes_le(262));
let b = BitAddress::of_token(&token_in_bytes_le(48_300));
println!("notebook-08 collision pin: 262LE and 48300LE share bit {} — {}", a.bit(), a == b);


759,0 -> bit 24288 (reg 759, tz 0)
geometry: P=10 M=1024 bits/reg=32
notebook-08 collision pin: 262LE and 48300LE share bit 24288 — true


---
## 2. The LUT lattice: nodes and fibers

A LUT is a labeled subset of `T` — a node of `2^T`. `K_i` is the fiber of
bit address `i`, and `LutIndex` is the fiber decomposition.


In [5]:
let lut: LutNode = LutNode::new("tokenLUT_1", (0..64u32).map(token_in_bytes));
let idx: LutIndex = LutIndex::build(&lut);
let i = BitAddress::of_token(&token_in_bytes(44)).bit();
let k_i: LutNode = LutNode::fiber("K_i", i, (0..64u32).map(token_in_bytes));
println!("LUT node '{}' has {} tokens; K_{} has {} tokens", lut.label, lut.tokens.len(), i, k_i.tokens.len());
println!("idx.fiber({}) has {} candidates", i, idx.fiber(i).len());
println!("ingest(K_{}) is the atom: popcount {} (must be 1)", i, k_i.to_hllset().popcount());


LUT node 'tokenLUT_1' has 64 tokens; K_13728 has 1 tokens
idx.fiber(13728) has 1 candidates
ingest(K_13728) is the atom: popcount 1 (must be 1)


---
## 3. Ingest: complete, single touch, in-module

Each token: 3 seeded hashes → 3 atoms set + 3 LUT fibers + 1 TF increment,
all in one pass.


In [6]:
let mut ingest: Ingest = Ingest::new();
let tokens: Vec<Vec<u8>> = (0..128u32).map(token_in_bytes).collect();
ingest.ingest_tokens(tokens.iter().map(|t| t.as_slice()));
println!("touched {} tokens (exactly once each)", ingest.touched);
for s in 0..3 {
    println!("  seed {}: sketch popcount {}, LUT fibers used by {} tokens",
        SEEDS[s], ingest.hllset(s).popcount(), tokens.len());
}
println!("TF(tid7) = {}", ingest.tf().count(&token_in_bytes(7)));


touched 128 tokens (exactly once each)
  seed 0: sketch popcount 125, LUT fibers used by 128 tokens
  seed 1: sketch popcount 123, LUT fibers used by 128 tokens
  seed 2: sketch popcount 127, LUT fibers used by 128 tokens
TF(tid7) = 1


---
## 4. Materialize: LUT-first, TF only for ambiguity

Candidates come from the pointed LUTs; TF is consulted only when a bit has
several candidates. A high-TF token absent from the LUTs never appears.


In [7]:
let mut ci: Ingest = Ingest::new();
let c1 = token_in_bytes_le(262).to_vec();
let c2 = token_in_bytes_le(48_300).to_vec();
ci.ingest_token(&c1);
ci.ingest_token(&c1); // TF(c1) = 2
ci.ingest_token(&c2); // TF(c2) = 1
let mut coll: HLLSet = HLLSet::new();
coll.add_bit(759 * 32 + 0);
let restored = materialize(&[(&coll, ci.lut(0))], ci.tf());
println!("collided bit (759,0) restores: {:?} (TF-max wins)", restored);
println!("restored == {{tid262LE}}: {}", restored == std::collections::BTreeSet::from([c1.clone()]));


collided bit (759,0) restores: {[6, 1, 0, 0]} (TF-max wins)
restored == {tid262LE}: true


---
## 5. The Noether context

`S(t)` with declared generators; `evolve` gives D/R/N with invariants; the
follow matrix is tropical (`max`, `+`), grow-only, with lattice projection.


In [8]:
let ga: HLLSet = HLLSet::from_tokens([token_in_bytes(0), token_in_bytes(1), token_in_bytes(2)]);
let gb: HLLSet = HLLSet::from_tokens([token_in_bytes(3), token_in_bytes(4)]);
let gc: HLLSet = HLLSet::from_tokens([token_in_bytes(2), token_in_bytes(5), token_in_bytes(6)]);
let previous: HLLSet = HLLSet::union_all(vec![ga.clone(), gb.clone()]);
let current: HLLSet = HLLSet::union_all(vec![ga.clone(), gc.clone()]);
let n = evolve(&previous, &current);
println!("D={} R={} N={}  invariants={}",
    n.departed.popcount(), n.retained.popcount(), n.novel.popcount(),
    invariants_hold(&n, &previous, &current));
let ctx: Context = Context::from_generators(vec![ga, gc]);
println!("S(t) = join of {} generators, popcount {}", ctx.generators().len(), ctx.state().popcount());
let mut fm: FollowMatrix = FollowMatrix::default();
fm.observe(b"a", b"b", 2); fm.observe(b"a", b"b", 5);
println!("follow(a,b) = {} (tropical max)", fm.get(b"a", b"b"));


D=2 R=3 N=2  invariants=true
S(t) = join of 2 generators, popcount 5
follow(a,b) = 5 (tropical max)


---
## 6. The five-level rank algebra

TF is stored; rank is derived: `F(TF) → G(bit) → H(register) → K(HLLSet) → L(compound)`.


In [9]:
let tf: hllset_morphisms::TfTable = ingest.tf().clone();
let lut0: LutIndex = ingest.lut(0).clone();
let sk: HLLSet = ingest.hllset(0).clone();
let f = token_rank(&tf, &token_in_bytes(7));
let first_bit = sk.bit_addresses()[0].bit();
let g = bit_rank(&tf, &lut0, first_bit, Aggregator::Sum);
let h = register_rank(&tf, &lut0, sk.bit_addresses()[0].reg(), Aggregator::Sum);
let k = hllset_rank(&tf, &lut0, &sk, Aggregator::Sum);
let l = compound_rank(&tf, &lut0, &[sk.clone()], Aggregator::Sum);
println!("F(tid7)={}  G(bit {})={}  H(reg {})={}  K(H)={}  L({{H}})={}",
    f, first_bit, g, sk.bit_addresses()[0].reg(), h, k, l);


F(tid7)=1  G(bit 97)=1  H(reg 3)=1  K(H)=128  L({H})=128


---
## 7. The finest symmetry

`HLLSet(LUT) = HLLSet(MerkleTree)`: ingesting the LUT equals joining its
atom tree — two presentations of one lattice element.


In [10]:
let sym_lut: LutNode = LutNode::new("tokenLUT_all", (0..300u32).map(token_in_bytes));
let from_tokens: HLLSet = sym_lut.to_hllset();
let tree: AtomTree = AtomTree::from_lut(&sym_lut);
let tree_atoms: Vec<u32> = from_tokens.bit_addresses().iter().map(|a| a.bit()).collect();
println!("ingest(LUT) atoms = {} ; MerkleTree atoms = {}", tree_atoms.len(), tree.atoms.len());
println!("same atoms: {}", tree_atoms == tree.atoms);
println!("finest symmetry holds: {}", tree_atoms == tree.atoms && tree.root().len() == 40);


ingest(LUT) atoms = 288 ; MerkleTree atoms = 288
same atoms: true
finest symmetry holds: true


---
## Summary

One hinge, one LUT lattice, two morphisms with their contracts, the
Noether context, derived ranks, and the finest symmetry — all as
executable gen2 code. The legacy notebooks stay discharged; this is the
first of the new set.
